In [0]:
SELECT * FROM workspace.default.final_quick_comm_dataset LIMIT 10

In [0]:
DESC workspace.default.final_quick_comm_dataset

In [0]:
SELECT
    COUNT(DISTINCT order_id) AS total_orders,
    ROUND(SUM(payment_value),2) AS GMV,
    ROUND(SUM(payment_value) / COUNT(DISTINCT order_id),2) AS AOV,
    ROUND(AVG(review_score),2) AS avg_review_score
FROM workspace.default.final_quick_comm_dataset;

In [0]:
WITH customer_orders AS (

    SELECT
        customer_unique_id,
        COUNT(DISTINCT order_id) AS order_count

    FROM workspace.default.final_quick_comm_dataset

    GROUP BY 1
)

SELECT
    CASE
        WHEN order_count = 1 THEN 'One-Time'
        ELSE 'Repeat'
    END AS customer_type,

    COUNT(*) AS customers

FROM customer_orders

GROUP BY 1;

In [0]:
WITH first_purchase AS (

    SELECT
        customer_unique_id,
        date(MIN(date_trunc('month', order_purchase_timestamp))) AS cohort_month

    FROM workspace.default.final_quick_comm_dataset

    GROUP BY 1
),

monthly_orders AS (

    SELECT
        customer_unique_id,
        date(date_trunc('month', order_purchase_timestamp)) AS order_month

    FROM workspace.default.final_quick_comm_dataset
)

SELECT
    fp.cohort_month,
    mo.order_month,
    COUNT(DISTINCT mo.customer_unique_id) AS active_customers

FROM first_purchase fp

JOIN monthly_orders mo
ON fp.customer_unique_id = mo.customer_unique_id

GROUP BY 1,2
ORDER BY 1,2;

In [0]:
SELECT

    CASE
        WHEN order_delivered_customer_date >
             order_estimated_delivery_date

        THEN 'Delayed'

        ELSE 'On-Time'
    END AS delivery_status,

    COUNT(DISTINCT order_id) AS orders,

    ROUND(AVG(review_score),2) AS avg_review_score,

    ROUND(AVG(payment_value),2) AS avg_order_value

FROM workspace.default.final_quick_comm_dataset

WHERE order_status = 'delivered'

GROUP BY 1;

In [0]:
SELECT

    product_category_name_english,

    COUNT(DISTINCT order_id) AS total_orders,

    ROUND(SUM(payment_value),2) AS revenue,

    ROUND(AVG(review_score),2) AS avg_review_score,

    ROUND(AVG(delivery_delay_days),2) AS avg_delay_days

FROM workspace.default.final_quick_comm_dataset

GROUP BY 1

ORDER BY revenue DESC
LIMIT 10;

In [0]:
SELECT

    product_category_name_english,

    COUNT(DISTINCT order_id) AS cancelled_orders

FROM workspace.default.final_quick_comm_dataset

WHERE order_status = 'canceled'
and product_category_name_english IS NOT NULL
GROUP BY 1
ORDER BY cancelled_orders DESC
LIMIT 10;

In [0]:
SELECT

    order_status,

    COUNT(DISTINCT order_id) AS orders

FROM workspace.default.final_quick_comm_dataset

GROUP BY 1

ORDER BY orders DESC;

In [0]:
-- What customer behaviors correlate with repeat purchases?
-- Compare:
-- repeat vs one-time buyers
-- AOV
-- delivery delays
-- review scores